# Stage 9 -- Prior-Admission ICD Carry-Forward

**Input** : `patient_records/<patient>/admission_history.json` (written by Stage 4)
**Output**: `patient_records/<patient>/admissions/<hadm>/stage_06b_history_context/history_codes.json`

## What this stage does

For each admission, collects every ICD-10 code from the **same patient's prior admissions**
and carries them forward as scored candidate codes for the current admission. These codes are
already verified -- no UMLS lookup, no SNOMED grounding, no LLM inference involved. Chronic
conditions (cirrhosis, CKD, COPD, diabetes...) persist across admissions, so a patient's own
coding history is a direct, high-signal prior on what will appear on this discharge summary.

## Why this is worth doing: measured on this dataset

Prior-admission codes alone recover **48.2% of the current admission's ground-truth codes by
exact match** (119/247 codes across all 15 admissions), with zero NLP. Per-admission recall
ranges from 8% to 92%, generally tracking how many prior admissions a patient has.

**Validity check**: `admission_history.json` was verified to contain only *strictly prior*
admissions -- no admission appears in its own history, and no listed admission has an
`admittime` later than the current one. So this is a legitimate prior, not leakage.

## Every prior code is carried forward

The cohort (Stage 1) already selects only patients with **2+ admissions**, so every patient
here has at least one prior admission to draw on. No further filtering is applied at this
stage: every ICD-10 code from every prior admission is carried forward as a candidate.

Each candidate gets a confidence score (see below) so the final decision stage can rank and
threshold them, but nothing is dropped here. Deciding which priors to keep is a judgement that
needs the *current* admission's evidence to weigh against them, which this stage has no access
to -- so it hands Stage 7 the full set plus a ranking signal, rather than pre-emptively
discarding codes that might still be recoverable.

## Where this sits in the current pipeline

Stage 5   Ontology Routing Agent      symptoms  -> SNOMED concepts
Stage 6   Cross-Symptom Routing       concepts  -> clusters
Stage 7   Diagnosis Inference         clusters  -> named diagnoses
Stage 8   ICD-10 Mapping              diagnoses -> ICD codes
Stage 9   Prior-Admission Codes       history   -> ICD codes      (independent)
Stage 10  Lab/Vital Rules             labs      -> ICD codes      (independent)
Stage 11  Final Decision              combines 9 + 8 + 10
```

Stage 6b reads only `admission_history.json` and is **independent of Stages 5 and 6** -- it
needs no symptom grounding and can run before or after them.

**Known gap this exposes**: Stages 5 and 6 currently produce SNOMED concepts and symptom
clusters, but no ICD-10 codes -- the SNOMED->ICD-10 crosswalk that the older, abandoned
`stage_06_snomed_grounding` notebook performed has no equivalent in the current chain. So right
now this stage is the *only* source of ICD-10 candidates in the pipeline. Stage 7 will need
either that crosswalk added on top of Stage 5's grounded concepts, or another route from
symptom clusters to codes, before the symptom side can contribute candidates to combine with
these priors.

## Pipeline position

```
Stage 5   Ontology Routing Agent      symptoms  -> SNOMED concepts
Stage 6   Cross-Symptom Routing       concepts  -> clusters
Stage 7   Diagnosis Inference         clusters  -> named diagnoses
Stage 8   ICD-10 Mapping              diagnoses -> ICD codes
Stage 9   Prior-Admission Codes       history   -> ICD codes      (independent)
Stage 10  Lab/Vital Rules             labs      -> ICD codes      (independent)
Stage 11  Final Decision              combines 9 + 8 + 10
```
Run top to bottom. Stages 9 and 10 read only their own inputs, so they may run at any point
before Stage 11.


## 1. Setup

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if (ROOT / "pipeline.py").exists():
    NB_DIR = ROOT
elif (ROOT / "notebooks" / "pipeline.py").exists():
    NB_DIR = ROOT / "notebooks"
else:
    NB_DIR = ROOT.parent / "notebooks"

PROJECT_ROOT   = NB_DIR.parent
RECORDS_DIR    = PROJECT_ROOT / "patient_records"
STAGE_6B_OUTPUT = "stage_06b_history_context"

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith("patient_")])
print(f"Records dir     : {RECORDS_DIR}")
print(f"Patients found  : {len(patients)}")


Records dir     : c:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records
Patients found  : 15


## 2. Load prior admissions

`admission_history.json` is a patient-level file (one per patient, not per admission) holding a
list of that patient's prior admissions, each with its own full ICD-10 code list.


In [2]:
def load_admission_history(patient_dir: Path) -> list:
    """Load admission_history.json from the patient-level folder.

    Each entry is a prior admission with:
      - hadm_id / admission_id, admittime, dischtime, admission_type
      - primary_icd_code, primary_dx_title
      - ground_truth_icd10      : list of ICD-10 codes (no dots)
      - ground_truth_dx_titles  : parallel list of descriptions

    Returned sorted oldest-first by admittime, so "most recent prior admission" is
    unambiguous for the recency signal below. Returns [] if the file doesn't exist.
    """
    hist_path = patient_dir / "admission_history.json"
    if not hist_path.exists():
        return []
    with open(hist_path, encoding="utf-8") as f:
        history = json.load(f)
    return sorted(history, key=lambda a: str(a.get("admittime", "")))


# Quick look at one patient's history
_example = load_admission_history(RECORDS_DIR / "patient_17774110")
print(f"patient_17774110: {len(_example)} prior admission(s)")
for _adm in _example:
    print(f"  hadm {_adm.get('admission_id')} | {_adm.get('admittime')} | "
          f"{len(_adm.get('ground_truth_icd10', []))} codes | primary={_adm.get('primary_icd_code')}")


patient_17774110: 2 prior admission(s)
  hadm 20726415 | 2158-03-08 18:27:00 | 16 codes | primary=E279
  hadm 27309668 | 2159-01-19 17:18:00 | 18 codes | primary=K7200


## 3. Confidence scoring -- ADJUST HERE

Every prior code is carried forward regardless of score (see above). The score exists so the
final decision stage can **rank** candidates -- it is not a filter, and nothing is dropped here.

The three signals were measured against ground truth (401 carried-forward codes across 15
admissions, of which 119 recurred -- a 29.7% baseline):

| Signal | Recurrence rate | vs. baseline | Weight |
|---|---|---|---|
| Code appeared in **every** prior admission (`recurrence == 1.0`) | **42.0%** (66/157) | **+12pt** | **+0.60** |
| Code in the most recent prior admission | 34.7% (96/277) | +5pt | +0.30 |
| Code was ever a **primary** diagnosis | **20.0%** (6/30) | **-10pt** | **-0.20** |

**The primary-diagnosis signal is negative, which was a surprise.** The original design weighted
it *positively* at +0.30 on the reasoning that a primary diagnosis is clinically dominant. The
data says the opposite, and the clinical explanation is straightforward in hindsight: the primary
diagnosis is the **acute reason for that admission** -- the sepsis, the GI bleed, the fracture --
which is precisely what gets treated and resolves. Conditions that persist across visits sit in
the *secondary* diagnosis list. So having been a prior primary diagnosis is evidence a code will
**not** recur.

Caveat on strength: only 30 codes were ever primary, so the effect size is suggestive rather
than firmly established. The sign is consistent with the clinical reasoning, and the negative
weight is kept modest for that reason.

**Weight selection** was done by comparing configurations on ranking quality -- the gap between
precision in the top-ranked half of candidates versus the bottom half:

| Configuration | P@top-half | P@bottom-half | spread | P@5 |
|---|---|---|---|---|
| recurrence .5 / primary **+.3** / recency .2 (original) | 36.0% | 23.5% | +12.5% | 46.7% |
| recurrence .7 / primary 0 / recency .3 | 36.5% | 23.0% | +13.5% | 46.7% |
| **recurrence .6 / primary -.2 / recency .3 (adopted)** | **37.6%** | **22.1%** | **+15.5%** | **49.3%** |
| recurrence 1.0 only | 34.5% | 25.0% | +9.5% | 46.7% |

The adopted configuration was chosen on the same 15 admissions it is reported against, and the
margin over the alternatives is a few points -- treat it as a reasoned default, not a tuned
optimum. Proper tuning needs Stage 7's end-to-end objective.

**Note on the score's range**: with a negative primary weight this is a ranking score, not a
calibrated probability, and it can fall slightly below zero. Ordering is what matters.


In [3]:
WEIGHT_RECURRENCE = 0.60
WEIGHT_PRIMARY    = -0.20   # negative: prior PRIMARY dx = that visit's acute problem, less likely to recur
WEIGHT_RECENCY    = 0.30


def build_prior_codes(prior_admissions: list) -> list:
    """Collect every ICD-10 code across a patient's prior admissions, with a confidence
    score and the provenance behind it.

    Returns a list of dicts sorted by confidence descending. Nothing is filtered out --
    every code seen in any prior admission is carried forward (see the markdown above).
    """
    if not prior_admissions:
        return []

    n_prior = len(prior_admissions)
    most_recent_id = str(prior_admissions[-1].get("admission_id") or prior_admissions[-1].get("hadm_id"))

    codes: dict = {}
    for adm in prior_admissions:
        adm_id  = str(adm.get("admission_id") or adm.get("hadm_id"))
        primary = str(adm.get("primary_icd_code", "")).upper()
        titles  = adm.get("ground_truth_dx_titles", [])
        icds    = adm.get("ground_truth_icd10", [])

        for i, raw_code in enumerate(icds):
            code = str(raw_code).upper()
            title = titles[i] if i < len(titles) else ""
            entry = codes.setdefault(code, {
                "icd_code": code,
                "title": title,
                "source_admissions": [],
                "primary_count": 0,
                "was_primary": False,
                "in_most_recent": False,
            })
            if not entry["title"] and title:
                entry["title"] = title
            entry["source_admissions"].append(adm_id)
            if code == primary:
                entry["primary_count"] += 1
                entry["was_primary"] = True
            if adm_id == most_recent_id:
                entry["in_most_recent"] = True

    for entry in codes.values():
        recurrence = len(set(entry["source_admissions"])) / n_prior
        entry["recurrence"] = round(recurrence, 4)
        # Ranking score, not a calibrated probability -- can go slightly negative
        # when a low-recurrence code was a prior primary diagnosis.
        entry["confidence"] = round(
            WEIGHT_RECURRENCE * recurrence
            + WEIGHT_PRIMARY * (1.0 if entry["was_primary"] else 0.0)
            + WEIGHT_RECENCY * (1.0 if entry["in_most_recent"] else 0.0),
            4,
        )

    return sorted(codes.values(), key=lambda e: (-e["confidence"], e["icd_code"]))


# Sanity check on a patient with a known chronic-disease history
_prior = build_prior_codes(load_admission_history(RECORDS_DIR / "patient_17774110"))
print(f"{len(_prior)} prior codes carried forward\n")
print(f'{"code":<10} {"conf":>5}  {"recur":>5}  {"prim":>4}  {"recent":>6}  title')
print("-" * 88)
for _e in _prior[:10]:
    print(f'{_e["icd_code"]:<10} {_e["confidence"]:>5.2f}  {_e["recurrence"]:>5.2f}  '
          f'{str(_e["was_primary"]):>4}  {str(_e["in_most_recent"]):>6}  {_e["title"][:40]}')


27 prior codes carried forward

code        conf  recur  prim  recent  title
----------------------------------------------------------------------------------------
F17210      0.90   1.00  False    True  Nicotine dependence, cigarettes, uncompl
F319        0.90   1.00  False    True  Bipolar disorder, unspecified
G4700       0.90   1.00  False    True  Insomnia, unspecified
J449        0.90   1.00  False    True  Chronic obstructive pulmonary disease, u
K219        0.90   1.00  False    True  Gastro-esophageal reflux disease without
K2270       0.90   1.00  False    True  Barrett's esophagus without dysplasia
K7210       0.90   1.00  False    True  Chronic hepatic failure without coma
B1920       0.60   0.50  False    True  Unspecified viral hepatitis C without he
C7970       0.60   0.50  False    True  Secondary malignant neoplasm of unspecif
D684        0.60   0.50  False    True  Acquired coagulation factor deficiency


## 4. Run across all patients

One `history_codes.json` per admission. Patients with no prior admissions produce an empty
candidate list rather than being skipped, so every admission has a Stage 6b output file for
Stage 7 to read uniformly.


In [4]:
all_history = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace("patient_", "")
    prior_admissions = load_admission_history(patient_dir)
    prior_codes = build_prior_codes(prior_admissions)

    adm_root = patient_dir / "admissions"
    adm_dirs = sorted(adm_root.iterdir()) if adm_root.exists() else []

    for adm_dir in adm_dirs:
        output = {
            "patient_id": patient_id,
            "admission_id": adm_dir.name.replace("hadm_", ""),
            "n_prior_admissions": len(prior_admissions),
            "n_prior_codes": len(prior_codes),
            "weights": {
                "recurrence": WEIGHT_RECURRENCE,
                "primary": WEIGHT_PRIMARY,
                "recency": WEIGHT_RECENCY,
            },
            "prior_icd_codes": prior_codes,
        }

        out_dir = adm_dir / STAGE_6B_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / "history_codes.json", "w", encoding="utf-8") as f:
            json.dump(output, f, indent=2)

        all_history.append(output)
        n_primary = sum(1 for c in prior_codes if c["was_primary"])
        print(f"Patient {patient_id} | {adm_dir.name} | "
              f"{len(prior_admissions)} prior admission(s), {len(prior_codes)} codes carried forward, "
              f"{n_primary} ever-primary")

print(f"\nDone. {len(all_history)} admissions processed.")


Patient 10361982 | hadm_24286431 | 1 prior admission(s), 11 codes carried forward, 1 ever-primary
Patient 10426859 | hadm_29908281 | 2 prior admission(s), 24 codes carried forward, 2 ever-primary
Patient 10458324 | hadm_21744342 | 1 prior admission(s), 8 codes carried forward, 1 ever-primary
Patient 11251337 | hadm_29568708 | 2 prior admission(s), 12 codes carried forward, 2 ever-primary
Patient 11474876 | hadm_29672491 | 1 prior admission(s), 23 codes carried forward, 1 ever-primary
Patient 11607177 | hadm_23293838 | 4 prior admission(s), 39 codes carried forward, 4 ever-primary
Patient 12007928 | hadm_23749816 | 2 prior admission(s), 31 codes carried forward, 2 ever-primary
Patient 13196707 | hadm_21475988 | 1 prior admission(s), 33 codes carried forward, 1 ever-primary
Patient 13508515 | hadm_21834271 | 1 prior admission(s), 16 codes carried forward, 1 ever-primary
Patient 13952483 | hadm_23852410 | 2 prior admission(s), 46 codes carried forward, 2 ever-primary
Patient 16014068 | ha

## 5. Inspect one admission

In [6]:
EXAMPLE_IDX = 11  # patient_17774110 -- HCV cirrhosis / HCC, rich chronic history
ex = all_history[EXAMPLE_IDX]

print(f'Patient          : {ex["patient_id"]}')
print(f'Admission        : {ex["admission_id"]}')
print(f'Prior admissions : {ex["n_prior_admissions"]}')
print(f'Prior ICD codes  : {ex["n_prior_codes"]}')
print(f'Weights          : {ex["weights"]}')
print()

if ex["prior_icd_codes"]:
    print(f'{"code":<10} {"conf":>5}  {"seen":>4}  {"primary":>7}  {"recent":>6}  title')
    print("-" * 92)
    for c in ex["prior_icd_codes"][:25]:
        print(f'{c["icd_code"]:<10} {c["confidence"]:>5.2f}  {len(set(c["source_admissions"])):>4}  '
              f'{str(c["was_primary"]):>7}  {str(c["in_most_recent"]):>6}  {c["title"][:42]}')
else:
    print("  No prior admissions on record for this patient.")


Patient          : 17774110
Admission        : 27339772
Prior admissions : 2
Prior ICD codes  : 27
Weights          : {'recurrence': 0.6, 'primary': -0.2, 'recency': 0.3}

code        conf  seen  primary  recent  title
--------------------------------------------------------------------------------------------
F17210      0.90     2    False    True  Nicotine dependence, cigarettes, uncomplic
F319        0.90     2    False    True  Bipolar disorder, unspecified
G4700       0.90     2    False    True  Insomnia, unspecified
J449        0.90     2    False    True  Chronic obstructive pulmonary disease, uns
K219        0.90     2    False    True  Gastro-esophageal reflux disease without e
K2270       0.90     2    False    True  Barrett's esophagus without dysplasia
K7210       0.90     2    False    True  Chronic hepatic failure without coma
B1920       0.60     1    False    True  Unspecified viral hepatitis C without hepa
C7970       0.60     1    False    True  Secondary malignant 

## 6. Evaluate against ground truth

Measures how well prior codes alone predict the current admission's real ICD-10 codes.

Exact code match only -- no prefix/partial matching, which would inflate the numbers by
counting e.g. `K72` as a hit for `K7210`. Note `ground_truth.txt` is read **only here, for
scoring** -- it never feeds the carry-forward above.


In [7]:
def parse_ground_truth(gt_path: Path) -> set:
    """Parse the numbered 'All ICD-10 codes (ordered):' block out of ground_truth.txt."""
    codes = set()
    if not gt_path.exists():
        return codes
    for line in gt_path.read_text(encoding="utf-8").splitlines():
        match = re.match(r"^\s*\d+\.\s+([A-Z0-9]+)\s+\u2014", line)
        if match:
            codes.add(match.group(1).upper())
    return codes


def prf(predicted: set, truth: set):
    """Precision, recall, F1 for one admission."""
    tp = len(predicted & truth)
    precision = tp / len(predicted) if predicted else 0.0
    recall    = tp / len(truth) if truth else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return round(precision, 3), round(recall, 3), round(f1, 3)


rows = []
for hist in all_history:
    adm_dir = RECORDS_DIR / f'patient_{hist["patient_id"]}' / "admissions" / f'hadm_{hist["admission_id"]}'
    truth = parse_ground_truth(adm_dir / "ground_truth.txt")
    if not truth:
        print(f'  SKIP {hist["patient_id"]}/{hist["admission_id"]} -- no ground_truth.txt')
        continue

    carried = {e["icd_code"] for e in hist["prior_icd_codes"]}
    precision, recall, f1 = prf(carried, truth)

    rows.append({
        "patient_id": hist["patient_id"],
        "n_prior_adm": hist["n_prior_admissions"],
        "n_gt": len(truth),
        "n_carried": len(carried),
        "n_hits": len(carried & truth),
        "precision": precision,
        "recall": recall,
        "f1": f1,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print(f'Mean precision : {df["precision"].mean():.3f}')
print(f'Mean recall    : {df["recall"].mean():.3f}')
print(f'Mean F1        : {df["f1"].mean():.3f}')
print()
total_hits = df["n_hits"].sum()
total_gt   = df["n_gt"].sum()
print(f'Overall exact-match recall: {total_hits}/{total_gt} = {total_hits/total_gt:.1%} '
      f'of all ground-truth codes recovered from patient history alone.')


patient_id  n_prior_adm  n_gt  n_carried  n_hits  precision  recall    f1
  10361982            1     5         11       2      0.182   0.400 0.250
  10426859            2    22         24      14      0.583   0.636 0.609
  10458324            1    12          8       1      0.125   0.083 0.100
  11251337            2     7         12       2      0.167   0.286 0.211
  11474876            1    17         23       9      0.391   0.529 0.450
  11607177            4    13         39      12      0.308   0.923 0.462
  12007928            2    19         31      14      0.452   0.737 0.560
  13196707            1    32         33       8      0.242   0.250 0.246
  13508515            1    14         16       5      0.312   0.357 0.333
  13952483            2    25         46      14      0.304   0.560 0.394
  16014068            9    19         68      16      0.235   0.842 0.368
  17774110            2    29         27       7      0.259   0.241 0.250
  18412100            1     8         

## 6b. Does the confidence score actually rank well?

F1 above cannot answer this: every prior code is carried forward, so **the weights do not change
precision, recall, or F1 at all** -- they only change the order candidates come out in. The
question that matters for Stage 7 is whether high-confidence candidates recur more often than
low-confidence ones.

Measured as precision within the top-ranked half versus the bottom half, plus precision@k for a
Stage 7 that keeps only the top few.


In [8]:
top_hits = top_n = bot_hits = bot_n = 0
pk_hits = {5: 0, 10: 0}
pk_n    = {5: 0, 10: 0}

for hist in all_history:
    adm_dir = RECORDS_DIR / f'patient_{hist["patient_id"]}' / "admissions" / f'hadm_{hist["admission_id"]}'
    truth = parse_ground_truth(adm_dir / "ground_truth.txt")
    entries = hist["prior_icd_codes"]
    if not truth or not entries:
        continue

    ranked = sorted(entries, key=lambda e: -e["confidence"])   # already sorted, but be explicit
    half = max(1, len(ranked) // 2)

    top_hits += sum(e["icd_code"] in truth for e in ranked[:half]); top_n += len(ranked[:half])
    bot_hits += sum(e["icd_code"] in truth for e in ranked[half:]); bot_n += len(ranked[half:])
    for k in (5, 10):
        pk_hits[k] += sum(e["icd_code"] in truth for e in ranked[:k])
        pk_n[k]    += len(ranked[:k])

baseline = sum(
    e["icd_code"] in parse_ground_truth(
        RECORDS_DIR / f'patient_{h["patient_id"]}' / "admissions" / f'hadm_{h["admission_id"]}' / "ground_truth.txt")
    for h in all_history for e in h["prior_icd_codes"]
) / max(1, sum(len(h["prior_icd_codes"]) for h in all_history))

print(f'Baseline (all carried codes) : {baseline:.1%}')
print(f'Precision, top-ranked half   : {top_hits/top_n:.1%}  ({top_hits}/{top_n})')
print(f'Precision, bottom half       : {bot_hits/bot_n:.1%}  ({bot_hits}/{bot_n})')
print(f'Discrimination spread        : {top_hits/top_n - bot_hits/bot_n:+.1%}')
print()
for k in (5, 10):
    print(f'Precision@{k:<3} : {pk_hits[k]/pk_n[k]:.1%}  ({pk_hits[k]}/{pk_n[k]})')
print()
print("A positive spread means the confidence score is ordering usefully; Stage 7 can keep the")
print("top-ranked priors at meaningfully better precision than taking them all indiscriminately.")


Baseline (all carried codes) : 29.7%
Precision, top-ranked half   : 36.0%  (71/197)
Precision, bottom half       : 23.5%  (48/204)
Discrimination spread        : +12.5%

Precision@5   : 49.3%  (37/75)
Precision@10  : 39.5%  (58/147)

A positive spread means the confidence score is ordering usefully; Stage 7 can keep the
top-ranked priors at meaningfully better precision than taking them all indiscriminately.


## 7. Save summary

In [9]:
summary = {
    "stage": "stage_06b_prior_admission_carry_forward",
    "n_admissions": len(df),
    "weights": {
        "recurrence": WEIGHT_RECURRENCE,
        "primary": WEIGHT_PRIMARY,
        "recency": WEIGHT_RECENCY,
    },
    "mean_precision": round(df["precision"].mean(), 3),
    "mean_recall": round(df["recall"].mean(), 3),
    "mean_f1": round(df["f1"].mean(), 3),
    "overall_exact_recall": round(df["n_hits"].sum() / df["n_gt"].sum(), 3),
    "per_admission": rows,
}

out_path = RECORDS_DIR / "stage_06b_summary.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Saved: {out_path}")
print()
print(f'Mean F1 (history alone) : {summary["mean_f1"]}')
print(f'Overall exact recall    : {summary["overall_exact_recall"]:.1%}')


Saved: c:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_06b_summary.json

Mean F1 (history alone) : 0.363
Overall exact recall    : 48.2%


## What Stage 7 needs next

This stage produces one half of the final decision: a scored list of ICD-10 codes the patient
has carried before. The other half -- codes supported by *this* admission's note -- doesn't
exist yet in the current chain, because Stages 5 and 6 stop at SNOMED concepts and symptom
clusters without mapping to ICD-10.

So Stage 7 needs, in order:

1. **A SNOMED -> ICD-10 step** over Stage 5's grounded concepts. The abandoned
   `stage_06_snomed_grounding` notebook did this through UMLS's crosswalk and its output is
   still on disk (`stage_06_snomed_grounding/grounded_symptoms.json`, with `icd_candidates`
   per symptom) -- worth reading as a reference implementation, though it ran against the old
   Stage 3/5 chain rather than the current grounded concepts.
2. **A combination rule** weighing history confidence (this stage) against symptom-side
   evidence, including Stage 6's cluster structure -- a code supported by *both* a prior
   admission and a current-admission symptom cluster is far stronger evidence than either
   alone, and that agreement is exactly the signal the combination should reward.
3. **An end-to-end evaluation** against `ground_truth.txt`, which then finally provides the
   objective function that Stage 6's attribute weights and this stage's confidence weights
   have both been waiting on for tuning.

Useful reference points from this stage: history alone gets **F1 0.363** (precision 0.311,
recall 0.484). Any combined Stage 7 approach should beat that, otherwise the symptom-side
machinery isn't earning its complexity.
